# Module 4 · Wiring Up the Agent (45 min)

The loop is **complete and read-only**. You supply three pieces in `ha_workshop/agent_config.py`:
the tools to register (**TODO-1**), the observation step (**TODO-2**), and the system prompt
(**TODO-3**). Read the loop, run the baseline, then fill the three TODOs.

In [ ]:
# Make the top-level ha_workshop package importable and anchor relative paths to the repo root.
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'healthagent').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print('repo root:', ROOT)

In [ ]:
import inspect
from healthagent.agent.loop import run_agent
print(inspect.getsource(run_agent))

### C2 · The shipped agent already runs (out of the box)
Only `compare_window` is wired, so it answers the **activity** question — but **not** the sleep
question yet. Making the sleep question work is your job (TODO-1).

In [ ]:
from healthagent.llm.client import get_client
from healthagent.agent.baseline import run_baseline_agent
client = get_client('scripted')   # deterministic teaching backend (everyone sees the same output)
print('ACTIVITY:', run_baseline_agent('How does my activity compare to my goal?', client=client).text)
print('\nSLEEP   :', run_baseline_agent('Why have I been sleeping poorly this week?', client=client).text)

In [ ]:
from ha_workshop import workshop_paths
print('Edit this file for Module 4:')
print('   ', workshop_paths()['agent_config'])

### Your three edits (in `ha_workshop/agent_config.py`)
- **TODO-1** `WORKSHOP_TOOLS = [retrieval.query_sleep, student_tools.query_screen_time, student_tools.detect_change, visualization.plot_timeseries]`
- **TODO-2** in `observe`, return `client.serialize_tool_result(tool_call, result)`
- **TODO-3** `SYSTEM_PROMPT = BASE_SYSTEM + '\n' + GROUNDING_CLAUSE + '\n' + REFUSAL_CLAUSE`

Save, then run the cell below (it reloads your config). Before the edits the agent answers
*"I could not gather enough evidence"* — that's the loop failing **safely**, not making things up.

In [ ]:
from ha_workshop import reload_ha_workshop
_, cfg = reload_ha_workshop()
for q in ['How does my activity compare to my goal?',
          'Why have I been sleeping poorly this week?']:
    ans = cfg.run(q, client=client)
    print('\nQ:', q)
    print('  tools used:', [t for s in ans.trace for t in s.get('tool_calls', [])])
    print('  grounded:', ans.grounded)
    print('  answer:', ans.text[:320])

In [ ]:
# Safety check: a medical-advice probe should be refused — but only after you add the
# [REFUSAL] clause in TODO-3.
print(cfg.run('Should I take melatonin for my sleep?', client=client).text)

### 🚀 Advanced (optional) — give the agent the confounds (`query_context`)

The agent blamed night screen time. But the dataset also seeds **confounds**: the `context` table has
a `deadline` event and a `caffeine_after_6pm` flag. A careful analyst asks: *what else changed this
week?*

Below is a complete `query_context` tool. Run it for u01's final week and read the confounds — and
notice their **coverage differs**: the `deadline` event spans the whole week, while `caffeine_after_6pm`
is true on only one day. So neither alone explains the sleep drop. That is exactly the
**correlation ≠ causation** point we discuss in Module 5.

**Your turn (live backend):** the code cell ends with an optional `HA_TRY_LIVE`-gated block that adds
`query_context` to the agent's tools and re-asks the sleep question — does the model now weigh the
confound? (On `scripted` the plan is fixed, so this needs a live model.)

In [ ]:
# Advanced (optional): expose the dataset's CONTEXT (confounds) as a tool.
from healthagent.tools.registry import tool
from healthagent.llm.schemas import ToolResult
from healthagent import data, config


@tool
def query_context(user_id: str, start_date: str, end_date: str) -> ToolResult:
    """Retrieve daily context (weekday/weekend/workday, weather, caffeine_after_6pm, any event) over an inclusive date range.

    Args:
        user_id: which user, e.g. 'u01'.
        start_date: ISO YYYY-MM-DD (inclusive).
        end_date: ISO YYYY-MM-DD (inclusive); 'this week' = the dataset's final 7 days.
    """
    cols = ['day_of_week', 'is_weekend', 'is_workday', 'weather', 'caffeine_after_6pm', 'event']
    df = data.load_user_metric(user_id, 'context', cols, start_date, end_date)   # filters user + dates
    rows = df.assign(date=df['date'].dt.strftime('%Y-%m-%d')).to_dict('records')
    n_caf = int(df['caffeine_after_6pm'].sum())
    events = sorted({e for e in df['event'].tolist() if e})
    return ToolResult(kind='table',
                      summary=f"{len(df)} days {start_date}..{end_date}: caffeine_after_6pm on {n_caf} day(s); events={events}.",
                      value=rows)


R0, R1 = config.recent_window()
res = query_context('u01', R0, R1)
print(res.summary)
for row in res.value:
    print('  ', row['date'], '| event:', row['event'], '| caffeine_after_6pm:', row['caffeine_after_6pm'])
print("\nNote: the 'deadline' event spans the whole week, but caffeine is true on only one day —"
      "\nneither alone explains the sleep drop. That is the correlation-vs-causation point for Module 5.")

# Your turn (LIVE only): let the agent SEE this tool and re-ask. cfg.run() rebuilds the registry every
# call via make_registry(), so add the tool to WORKSHOP_TOOLS (a one-off reg.register(...) won't take).
import os
if os.getenv('HA_TRY_LIVE') == '1':
    cfg.WORKSHOP_TOOLS = [*cfg.WORKSHOP_TOOLS, query_context]
    print('\nLIVE:', cfg.run('Why have I been sleeping poorly this week?', client=get_client('auto')).text[:400])
else:
    print('\n(Set HA_TRY_LIVE=1 and configure a live backend to let the agent use query_context.)')

### Trying a real model: free reasoning, *same* guardrails

Switch to a live backend (next cell, `HA_TRY_LIVE=1`) and the **model itself** plans and selects tools
— so it can answer open-ended questions, not just the scripted demos. But it runs through the **exact
same loop you just built**, so the same constraints still hold:

- **Your tools are the boundary** — it can only call the tools you registered in TODO-1; it can't reach
  data you didn't expose as a tool.
- **Grounding still gates the answer** — your `[GROUNDING]` clause + the loop's check (TODO-3); a claim
  with no matching evidence is not marked grounded.
- **Refusal is loop-owned and deterministic** — the medical-advice gate fires *before* the model is
  even called, so a live model **can't be talked into** giving medical advice either.
- **It still fails safe** — if the required evidence isn't gathered, you get the deterministic
  "couldn't gather enough evidence" answer, not a confident guess.

So a real model gives you **open-ended reasoning**, wrapped in the **same tools + safety rails** you
wired up here.

In [ ]:
# Optional — run YOUR agent on a LIVE backend (set HA_TRY_LIVE=1 and configure a key/Ollama).
# Same loop + reliability plumbing as the scripted run; it just swaps in a real model.
import os
if os.getenv('HA_TRY_LIVE') == '1':
    live = get_client('auto')
    print('live tier:', live.provider)
    ans = cfg.run('Why have I been sleeping poorly this week?', client=live)
    print('grounded:', ans.grounded, '| answer:', ans.text[:320])
else:
    print("Skipping live backend cell (set HA_TRY_LIVE=1 to try a live backend).")

### Discussion (no code): single agent vs. orchestrator-with-specialists
```
        ┌──────────── single agent (this tutorial / Part 1) ───────────┐
  user → plan → pick tool → run → observe → … → grounded answer

        ┌──── multi-agent orchestrator (Part 2 / GLOSS) ────┐
  user → planner ─┬─ retrieval agent ─┐
                  ├─ analysis agent  ─┼─ synthesizer → answer
                  └─ viz agent       ─┘
```
Multi-agent buys parallelism and specialization at the cost of coordination, latency, and more
failure modes — the focus of the companion GLOSS tutorial.